# Hill-Valley — експериментальне дослідження класифікації

Цей Notebook містить **лише розділи 1–2** роботи: формулювання експериментальної задачі та планування експерименту. Фактичні результати моделей і одноразове оцінювання на test set виконуються на наступному етапі.

## 1. Формулювання експериментальної задачі

### 1.1. Прикладна мета класифікації

Набір **Hill-Valley** містить записи, кожен з яких представляє 100 точок двовимірного графіка. Якщо точки побудувати послідовно, вони утворюють або **Hill** — підйом/«горб», або **Valley** — западину. Отже, прикладна мета класифікації полягає у визначенні типу форми за 100 числовими значеннями ознак. UCI визначає цю задачу як бінарну класифікацію; цільова змінна `class` має значення `0 = valley` та `1 = hill`. У наборі немає пропущених значень. citeturn0view0

**Тип задачі:** бінарна класифікація.

**Ознаки:** `X01 ... X100`, дійсні числові значення.

**Цільова змінна:** `class`.

**Класи:** `0 — Valley`, `1 — Hill`.

UCI описує Hill-Valley як sequential dataset із 606 записами та 101 змінною у представленні набору; 100 змінних є числовими ознаками, а `class` — цільовою змінною. citeturn0view0

### 1.2. Які помилки важливі

Для цієї задачі обидва класи є суттєвими: помилка `Valley → Hill` означає хибне визначення западини як пагорба, а `Hill → Valley` — хибне визначення пагорба як западини. Оскільки в умові не задано різної вартості цих двох помилок, у базовому експерименті вони розглядаються як однаково важливі.

Тому додатково до загальної якості потрібно аналізувати **confusion matrix**, precision, recall та F1-score для обох класів.

### 1.3. Основна метрика порівняння

Основною метрикою буде **macro F1-score (`f1_macro`)**. Для бінарної задачі він однаково враховує якість класифікації класів `Valley` і `Hill`, тому не дозволяє якості одного класу повністю приховати гіршу якість іншого.

**Додаткові метрики:** accuracy, precision, recall та confusion matrix. Також фіксуються час навчання і час прогнозування як інженерні показники.

### 1.4. Перевірювані гіпотези

| Гіпотеза | Експеримент | Критерій підтвердження |
|---|---|---|
| **H1. Масштабування ознак `StandardScaler` покращує якість KNN та SVM на Hill-Valley.** | Для KNN, Linear SVM і RBF SVM порівняти варіанти без масштабування та з `StandardScaler` за однаковою 5-fold Stratified CV. | Гіпотеза підтверджується, якщо середній CV `f1_macro` масштабованого варіанта вищий щонайменше на **0.01** хоча б для відповідного сімейства моделі без неприйнятного зростання інженерних витрат. |
| **H2. Метрика відстані та спосіб голосування впливають на якість KNN.** | Перевірити `metric ∈ {euclidean, manhattan}` та `weights ∈ {uniform, distance}` разом із фіксованою сіткою `n_neighbors`. | Гіпотеза підтверджується, якщо найкраща конфігурація KNN дає приріст середнього CV `f1_macro` не менше **0.01** відносно базової конфігурації `metric=euclidean, weights=uniform` за порівнюваного масштабування. |
| **H3. RBF SVM забезпечує вищу якість, ніж Linear SVM, для Hill-Valley.** | Порівняти Linear SVM та RBF SVM за однаковою 5-fold Stratified CV та заздалегідь визначеними сітками `C` і `gamma`. | Гіпотеза підтверджується, якщо найкращий RBF SVM має середній CV `f1_macro` щонайменше на **0.01** вищий за найкращий Linear SVM. |

Порівняння проводиться за однаковою процедурою оцінювання. Формулювання «спробувати різні моделі» саме по собі не є перевірюваною гіпотезою.

### 1.5. Інженерні критерії

Для Hill-Valley додатково враховуються:

- **затримка прогнозування** — час прогнозування одного набору або всієї тестової вибірки;
- **пам'ять / розмір моделі** — ресурсні витрати збереженої моделі;
- **пояснюваність** — можливість пояснити, які властивості 100 точок впливають на рішення;
- **вартість повторного навчання** — час, необхідний для повторного `fit` моделі.

## 2. Планування експерименту

До запуску пошуку гіперпараметрів фіксуються split, random state, схема CV, метрики, кандидати, способи масштабування, сітки та правило вибору фінальної моделі.

### 2.1. Джерело та спосіб отримання даних

Джерело даних — **UCI Machine Learning Repository, Hill-Valley, dataset ID 166**. UCI надає Python-приклад отримання даних через `ucimlrepo.fetch_ucirepo(id=166)`, після чого ознаки доступні як `hill_valley.data.features`, а цільова змінна — як `hill_valley.data.targets`. citeturn0view0

У цьому експерименті для методично контрольованого пошуку використовується завантажений набір `X, y`, після чого він один раз розділяється на training та test із фіксованими параметрами. Окремі UCI-файли `Training.data` та `Testing.data` також існують для варіантів із шумом і без шуму; UCI описує їх як окремі пари training/testing. citeturn0view0

**Важливо:** якщо в подальшій роботі буде обрано саме офіційні UCI Training/Testing файли, це потрібно зафіксувати як окремий варіант експериментального дизайну, а не непомітно змінювати поточний split.

### 2.2. Train/test split і random state

- **Train:** 80% даних.
- **Test:** 20% даних.
- **random_state:** `42`.
- `stratify=y` — для збереження пропорцій класів.
- Test set не використовується для вибору гіперпараметрів.

Таким чином, тестова вибірка відкривається лише один раз після завершення пошуку.

### 2.3. Stratified cross-validation

Для пошуку використовується **StratifiedKFold(n_splits=5, shuffle=True, random_state=42)**.

Стратифікація зберігає приблизно однакове співвідношення `Hill/Valley` у кожному fold. Усі кандидатні конфігурації оцінюються на однакових CV-розбиттях.

### 2.4. Основна та додаткові метрики

**Основна:** `f1_macro`.

**Додаткові:**
- accuracy;
- precision;
- recall;
- confusion matrix;
- training time;
- prediction time.

Основне правило порівняння — максимізація середнього `f1_macro` за 5-fold Stratified CV.

### 2.5. Кандидатні моделі

Для перевірки сформульованих гіпотез фіксуються три сімейства:

1. **KNN** — перевірка впливу масштабу, метрики відстані та способу голосування.
2. **Linear SVM** — лінійна межа класифікації.
3. **RBF SVM** — нелінійна межа класифікації.

### 2.6. Способи масштабування

Порівнюються два варіанти:

1. **без масштабування**;
2. **`StandardScaler`**.

Для варіанта з масштабуванням `StandardScaler` розміщується всередині `Pipeline`. Це запобігає використанню статистик test/fold validation під час навчання scaler.

### 2.7. Початкові сітки гіперпараметрів

**KNN:**
- `n_neighbors = [3, 5, 7, 11, 15]` — 5 значень;
- `weights = ['uniform', 'distance']` — 2 значення;
- `metric = ['euclidean', 'manhattan']` — 2 значення.

Разом: **20 конфігурацій**.

**Linear SVM:**
- `C = [0.01, 0.1, 1, 10, 100]` — **5 конфігурацій**.

**RBF SVM:**
- `C = [0.1, 1, 10, 100]` — 4 значення;
- `gamma = ['scale', 0.001, 0.01, 0.1]` — 4 значення.

Разом: **16 конфігурацій**.

Якщо кожна модель перевіряється з обома способами масштабування, загальна кількість кандидатних запусків подвоюється.

### 2.8. Правило вибору фінальної моделі

1. Вибирається конфігурація з максимальним середнім CV `f1_macro`.
2. Якщо різниця між кандидатами менша за **0.01**, додатково враховуються prediction latency, training time, пам'ять та пояснюваність.
3. Після фіксації фінальної конфігурації модель перенавчається на всій training-вибірці.
4. **Test set відкривається одноразово після завершення пошуку.**
5. Test-результат не використовується для подальшого підбору параметрів.

### 2.9. Правило зміни сітки після першого запуску

Якщо після першого запуску потрібно змінити сітку, початкові результати не видаляються. Зміна оформлюється новою перевірюваною гіпотезою із зазначенням:

- початкової сітки;
- нової сітки;
- причини зміни;
- очікуваного ефекту;
- критерію підтвердження.

Наприклад: «Перший запуск показав, що оптимальний `C` знаходиться на верхній межі початкової сітки. Тому перевіряється гіпотеза, що збільшення `C` до `[1000, 10000]` покращить CV `f1_macro` щонайменше на 0.01».

In [1]:
# Зафіксовані параметри експерименту — НЕ змінювати після першого запуску
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 5
SCORING = 'f1_macro'

KNN_GRID_SIZE = 5 * 2 * 2
LINEAR_SVM_GRID_SIZE = 5
RBF_SVM_GRID_SIZE = 4 * 4

print('Dataset: UCI Hill-Valley (ID 166)')
print('random_state:', RANDOM_STATE)
print('test_size:', TEST_SIZE)
print('CV folds:', N_SPLITS)
print('primary metric:', SCORING)
print('KNN configurations:', KNN_GRID_SIZE)
print('Linear SVM configurations:', LINEAR_SVM_GRID_SIZE)
print('RBF SVM configurations:', RBF_SVM_GRID_SIZE)

Dataset: UCI Hill-Valley (ID 166)
random_state: 42
test_size: 0.2
CV folds: 5
primary metric: f1_macro
KNN configurations: 20
Linear SVM configurations: 5
RBF SVM configurations: 16
